In [1]:
# [Claude] setup: run from repo root, imports
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
import numpy as np
import pandas as pd
import duckdb

In [2]:
conn = duckdb.connect('data/loans.duckdb', read_only=True)
defaults = conn.sql("SELECT funded_amnt, total_rec_prncp, recoveries, collection_recovery_fee FROM loans_scoped WHERE default_flag = 1 AND issue_year BETWEEN 2013 AND 2014").df()

defaults["net_loss"] = defaults["funded_amnt"] - defaults["total_rec_prncp"]-defaults["recoveries"]+defaults["collection_recovery_fee"]
lgd = defaults["net_loss"].sum() / defaults["funded_amnt"].sum()
print(f"Number of defaulted loans: {defaults.shape[0]}")
print(f"LGD: {lgd}")

Number of defaulted loans: 34693
LGD: 0.5063861502640687


In [3]:
scored = pd.read_parquet('data/processed/scored_test.parquet')
scored["el"] = scored["pd_lr"] * lgd * scored["funded_amnt"]

print(f"Total funded: {round(float(scored["funded_amnt"].sum() / 1e6),1)}")
print(f"Total expected loss: {round(float(scored["el"].sum() / 1e6),1)}")
print(f"Estimated loss as %: {round(float(scored["el"].sum() / scored["funded_amnt"].sum() * 100),1)}%")

Total funded: 3620.6
Total expected loss: 242.3
Estimated loss as %: 6.7%


In [6]:
loans = conn.sql("SELECT id, total_rec_int, total_rec_prncp, recoveries, collection_recovery_fee FROM loans").df()
loans_merged = scored.merge(loans, on="id", how="left")

print(f"Number of rows: {loans_merged.shape[0]}")
loans_merged['actual_loss'] = (loans_merged['funded_amnt'] - loans_merged['total_rec_prncp']-loans_merged['recoveries']+loans_merged['collection_recovery_fee']) * loans_merged['default_flag']
loans_merged['profit'] = loans_merged['total_rec_int']-loans_merged['actual_loss']

Number of rows: 282787


In [8]:
rows = []
for cutoff in [x/100 for x in range(5,41)]:
    approved = loans_merged[loans_merged["pd_lr"] < cutoff]
    rows.append({
        "cutoff": cutoff,
        "approval_rate": len(approved) / len(loans_merged),
        "default_rate": approved["default_flag"].mean(),
        "loss_mm": approved["actual_loss"].sum() / 1e6,
        "profit_mm": approved["profit"].sum() / 1e6
    })

cutoff_table = pd.DataFrame(rows)
print(cutoff_table.round(3))

    cutoff  approval_rate  default_rate  loss_mm  profit_mm
0     0.05          0.099         0.036    7.756     32.549
1     0.06          0.144         0.043   12.775     47.270
2     0.07          0.196         0.049   19.329     63.530
3     0.08          0.251         0.057   28.130     79.756
4     0.09          0.309         0.064   38.105     96.247
5     0.10          0.368         0.070   49.443    112.396
6     0.11          0.425         0.076   60.893    128.537
7     0.12          0.481         0.082   73.805    143.087
8     0.13          0.534         0.087   87.121    156.258
9     0.14          0.584         0.093  100.723    167.969
10    0.15          0.631         0.098  113.954    179.229
11    0.16          0.674         0.102  127.113    188.347
12    0.17          0.713         0.107  140.185    196.200
13    0.18          0.748         0.111  152.740    202.951
14    0.19          0.780         0.115  164.918    208.317
15    0.20          0.808         0.119 

In [16]:
import json

summary = []
for funding_cost_rate in [0.03, 0.05, 0.07]:
    loans_merged["net_profit"] = (
        loans_merged["profit"]
        - loans_merged["funded_amnt"] * funding_cost_rate
    )

    rows = []
    for cutoff in [x / 100 for x in range(5, 41)]:
        approved = loans_merged[loans_merged["pd_lr"] < cutoff]
        rows.append({
            "cutoff": cutoff,
            "approval_rate": len(approved) / len(loans_merged),
            "default_rate": approved["default_flag"].mean(),
            "net_profit_mm": approved["net_profit"].sum() / 1e6
        })

    cutoff_table_net = pd.DataFrame(rows)
    best = cutoff_table_net.loc[cutoff_table_net["net_profit_mm"].idxmax()]
    all_approved_net_profit = loans_merged["net_profit"].sum() / 1e6
    summary.append({
        "funding_cost": funding_cost_rate,
        "best_cutoff": best["cutoff"],
        "approval_rate": best["approval_rate"],
        "default_rate": best["default_rate"],
        "best_net_profit_mm": best["net_profit_mm"],
        "approve_all_net_profit_mm": all_approved_net_profit,
    })

sensitivity_table = pd.DataFrame(summary)
print(sensitivity_table.round(3))

sensitivity_table.to_csv("outputs/cutoff_sensitivity.csv", index=False)

funding_cost_rate = 0.05
loans_merged["net_profit"] = (
    loans_merged["profit"]
    - loans_merged["funded_amnt"] * funding_cost_rate
)
rows = []
for cutoff in [x / 100 for x in range(5, 41)]:
    approved = loans_merged[loans_merged["pd_lr"] < cutoff]
    rows.append({
        "cutoff": cutoff,
        "approval_rate": len(approved) / len(loans_merged),
        "default_rate": approved["default_flag"].mean(),
        "loss_mm": approved["actual_loss"].sum() / 1e6,
        "profit_mm": approved["profit"].sum() / 1e6,
        "net_profit_mm": approved["net_profit"].sum() / 1e6,
    })

cutoff_table_net = pd.DataFrame(rows)
cutoff_table_net.to_csv("outputs/cutoff_analysis.csv", index=False)

recommended = sensitivity_table.loc[sensitivity_table["funding_cost"] == 0.05].iloc[0]
recommended_approved = loans_merged[
    loans_merged["pd_lr"] < recommended["best_cutoff"]
]
actual_loss_mm = loans_merged["actual_loss"].sum() / 1e6
loss_reduction_pct = (
    (loans_merged["actual_loss"].sum() - recommended_approved["actual_loss"].sum())
    / loans_merged["actual_loss"].sum()
    * 100
)

with open("outputs/metrics.json") as f:
    metrics = json.load(f)
metrics["business"] = {
    "lgd": float(lgd),
    "total_el_mm": float(scored["el"].sum() / 1e6),
    "actual_loss_mm": float(actual_loss_mm),
    "recommended_cutoff": float(recommended["best_cutoff"]),
    "approval_rate": float(recommended["approval_rate"]),
    "loss_reduction_pct": float(loss_reduction_pct),
}
with open("outputs/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

   funding_cost  best_cutoff  approval_rate  default_rate  best_net_profit_mm  \
0          0.03         0.25          0.908         0.132             127.483   
1          0.05         0.18          0.748         0.111              63.756   
2          0.07         0.13          0.534         0.087              12.841   

   approve_all_net_profit_mm  
0                    120.863  
1                     48.450  
2                    -23.963  


**Summary**
At a base funding cost of 5%, the cutoff that maximizes net profit is 18%, leading to an approval rate of around 74.8% and lowering the default rate from 14.9% (when approving all) to 11.1%. The losses under this system are decreased by 44% ($273.8M → $152.7M) and net profit increases by 32% ($48.4M to $63.8M). At a higher funding cost, the optimal cutoff decreases (3% → 25% cutoff, 7% to 13% cutoff), and at 7% approving everyone loses money ($24.0M loss) while the 18%, 13% cutoff remains profitable ($12.8M).

**Assumptions:** Profit = interest received − net principal loss − funding cost (a flat % of funded amount). It excludes servicing costs, taxes, and capital charges, and is a backtest on 2015 loans.